# 🚀 Few-Shot Cross-Domain 3D Multi-Organ Segmentation
## Fault-Tolerant Benchmark Runner (Google Colab & Kaggle Free Tier)

### 🛡️ Built-in Free-Tier Protections:
1. **Automatic Checkpointing (`checkpoint_latest.pth` & `checkpoint_best.pth`):** Saves after every single epoch.
2. **Seamless Auto-Resume:** If your Colab/Kaggle session disconnects, times out, or gets preempted, re-running resumes from the exact last epoch without losing progress.
3. **Persistent Result Caching:** Completed runs are logged to `results/benchmark_summary.json` and `results/benchmark_summary.csv`. Already-evaluated regimes are skipped automatically to save your GPU quota!
4. **On-Demand Dataset Streaming:** AMOS 2022 and BTCV are streamed via Hugging Face without manual dataset downloads.

### Step 1: Environment & GPU Verification

In [ ]:
# Install lightweight dependencies
!pip install -q nibabel huggingface_hub scipy tqdm pandas matplotlib

import torch
print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Execution Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ GPU not detected. Please enable GPU in Runtime -> Change runtime type (Colab) or Accelerator -> GPU P100 (Kaggle).")

### Step 2: (Optional for Google Colab) Mount Google Drive for Permanent Storage
Uncomment the cell below if you are on Google Colab and want checkpoints and CSV tables saved directly to your Google Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_DIR = '/content/drive/MyDrive/Hybrid_Swin_UNet_Results'
# import os
# os.makedirs(DRIVE_DIR, exist_ok=True)
# print("Drive output directory ready:", DRIVE_DIR)

### Step 3: Load Workspace Code
Clones or navigates to the repository.

In [ ]:
import os, sys
if not os.path.exists("src"):
    !git clone https://github.com/pronob002/Hybrid_Swin_UNet.git
    %cd Hybrid_Swin_UNet

sys.path.append(os.getcwd())
print("Working directory:", os.getcwd())

### Step 4: Run 5-Shot Adaptation Benchmark
Runs the **Proposed Hybrid Swin-UNet** and **3D U-Net Baseline** with automated checkpointing and auto-resume.

In [ ]:
# 1. Hybrid Swin-UNet (Proposed)
!python train.py --model hybrid_swin --shots 5 --epochs 50 --eval_cases 10 --seed 42

# 2. 3D U-Net Baseline
!python train.py --model unet3d --shots 5 --epochs 50 --eval_cases 10 --seed 42

### Step 5: Multi-Regime Benchmark ($k=1, 3, 5, 10$ Shots & Multi-Seed Loop)
This loop automatically checks the result cache and skips already-completed runs if interrupted.

In [ ]:
from scripts.train_fewshot import run_experiment

regimes = [1, 3, 5, 10]
models = ["hybrid_swin", "unet3d"]
seeds = [42, 123, 456]

for seed in seeds:
    for m in models:
        for k in regimes:
            print(f"\n{'='*70}")
            print(f">>> Running {m.upper()} | {k}-Shot Adaptation | Seed: {seed}")
            print(f"{'='*70}")
            run_experiment(
                model_type=m,
                k_shots=k,
                num_epochs=50,
                seed=seed,
                eval_cases=10,
                checkpoint_base_dir="checkpoints",
                results_dir="results",
                force_rerun=False  # Skips completed experiments automatically!
            )

### Step 6: View Final Benchmark Results Table

In [ ]:
import pandas as pd
csv_path = "results/benchmark_summary.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("\n--- Completed Benchmark Experiments ---")
    display(df)
else:
    print("No results CSV found yet.")